In [ ]:
if __name__ != "__main__":
    raise Exception("This file is not meant to be imported")

In [ ]:
import ast
import sacrebleu
import json
from pathlib import Path
from pythainlp.tokenize import word_tokenize
from nltk.translate.nist_score import corpus_nist
from tqdm import tqdm

import os
def _get_basedir_path():
    match os.uname()[1]:
        case "lectrans":
            return Path("/large/datas/frederikus-h/lectrans-testset")
        case u if u.startswith("elm") or u.startswith("pine"):
            return Path(
                "~/_self_research/24.10.lecture_translate/lectrans-testset"
            ).expanduser()
        case _:
            raise NotImplementedError


def correcting_lang_id(lang):
    return {
        "jp": "ja"
    }.get(lang, lang)


def tokenize_thai(text):
    return " ".join(word_tokenize(text, engine="newmm"))


def get_bleu_kwargs(lang):
    kwargs = {}
    match lang:
        case "ja":
            kwargs["tokenize"] = "ja-mecab"
        case "zh":
            kwargs["tokenize"] = "zh"
        case "th":
            kwargs["tokenize"] = "none"
    return kwargs


def get_tokenized(text, language):
    match language:
        case "th":
            f_tokenize = tokenize_thai
        case "ja":
            f_tokenize = sacrebleu.metrics.BLEU(tokenize="ja-mecab").tokenizer
        case "zh":
            f_tokenize = sacrebleu.metrics.BLEU(tokenize="zh").tokenizer
        case _:
            f_tokenize = sacrebleu.metrics.BLEU(tokenize="13a").tokenizer
    return f_tokenize(text or "").split()


comet_model = None
cometkiwi_model = None
def comet_predict(preds, golds, srcs, batch_size=128):
    global comet_model, cometkiwi_model
    if comet_model is None or cometkiwi_model is None:
        from comet import download_model, load_from_checkpoint
        model_path = download_model("Unbabel/wmt22-comet-da")
        modelqe_path = download_model("Unbabel/wmt22-cometkiwi-da")
        print("COMET models downloaded")
        comet_model = load_from_checkpoint(model_path)
        cometkiwi_model = load_from_checkpoint(modelqe_path)
        assert comet_model and cometkiwi_model, "COMET models not loaded"
        print("COMET models loaded")
    data, data_qe = [], []
    for src, pred, gold in zip(srcs, preds, golds):
        data.append({"src": src,"mt": pred, "ref": gold})
        data_qe.append({"src": src,"mt": pred})
    score = comet_model.predict(data, batch_size=batch_size)
    score_qe = cometkiwi_model.predict(data_qe, batch_size=batch_size)
    return score, score_qe


In [ ]:
# import MeCab
# from rouge_score import rouge_scorer, tokenizer

# scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# class MecabTokenizer(tokenizer.Tokenizer):
#     def __init__(self, dic_path=None):
#         self.tagger = MeCab.Tagger('-Owakati')

#     def tokenize(self, text):
#         return self.tagger.parse(text).split()

# scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], tokenizer=MecabTokenizer())
# scores = scorer.score(ref, pred)


##### Computing Scores

In [ ]:
def _get_triplets_transcript_(src, gold, pred, lang):
    global sbj_lecid_pair, instance_id    # `instance_id` is timestamp
    if (
        (sbj_lecid_pair, instance_id) == ('intro_to_physics_20century/Fce9xf_kjyc', '0:10:08.636') and
        isinstance(preds[sbj_lecid_pair][instance_id], str)
    ):
        pred = preds[sbj_lecid_pair][instance_id]    # either parsing error or answer error, treating return as language-agnostic
    else:
        pred = preds[sbj_lecid_pair][instance_id].get(lang, "")
    if isinstance(pred, list):
        if len(pred) > 1:
            print(len(pred), instance_id, lang, sbj)
            pred = " ".join(pred)
        else:
            pred = pred[0] if len(pred) else ""
    gold_tokenized = get_tokenized(gold, lang)
    pred_tokenized = get_tokenized(pred, lang)
    if lang == "th":
        gold = " ".join(gold_tokenized)    # tokenize_thai(gold)
        pred = " ".join(pred_tokenized)    # tokenize_thai(pred)
    pred = '""' if (pred is None) or (len(pred) == 0) else pred
    if isinstance(pred, list) and len(pred) == 1:
        pred = pred[0]
    if isinstance(pred, list):
        pred = '\n'.join(pred)
    return [src], [gold], [pred], [gold_tokenized], [pred_tokenized]


def _get_triplets_slide_(all_bb_src, all_bb_gold, all_bb_pred, lang):
    global sbj_lecid_pair, instance_id    # `instance_id` is scene_id
    # currently, we assume the sequential order after sorting are always aligned
    _src = ['\n'.join(bb_text_list) for bb_key, bb_text_list in sorted(all_bb_src.items())]
    _gold = ['\n'.join(bb_text_list) for bb_key, bb_text_list in sorted(all_bb_gold.items())]
    _pred = ['\n'.join(bb_text_list) for bb_key, bb_text_list in sorted(all_bb_pred.items())]
    _gold_tokenized = [get_tokenized(_, lang) for _ in _gold]
    _pred_tokenized = [get_tokenized(_, lang) for _ in _pred]
    if lang == "th":
        _gold = [' '.join(_) for _ in _gold_tokenized]    # tokenize_thai(gold)
        _pred = [' '.join(_) for _ in _pred_tokenized]    # tokenize_thai(pred)
    return _src, _gold, _pred, _gold_tokenized, _pred_tokenized


def get_scores(golds, preds, compute_comet=False, batch_size=128, task="transcripts"):
    global \
        sbj_lecid_pair, cur_gold, scores_per_subject,\
        instance_id, gold_all_langs, scores_per_span, scores_per_lang,\
        lang, src, gold, pred, kwargs, score, score_qe,\
        gold_tokenized, pred_tokenized

    scores_all = {}
    for sbj_lecid_pair, cur_gold in golds.items():
        sbj, lec_id = sbj_lecid_pair.split('/', 1)

        # Debugging only
        # if sbj_lecid_pair not in preds:
        #     continue

        scores_per_subject = {
            "per_language": {},
            "per_instance": {},
        }
        scores_per_lang = {}
        for instance_id, gold_all_langs in cur_gold.items():
            scores_per_span = {}
            gold_all_langs = {correcting_lang_id(k): v for k, v in gold_all_langs.items()}
            for lang, gold in gold_all_langs.items():
                lang = correcting_lang_id(lang)
                if lang == 'en':
                    continue
                match task:
                    case "transcripts":
                        src, gold, pred, gold_tokenized, pred_tokenized = \
                            _get_triplets_transcript_(gold_all_langs['en'], gold, preds[sbj_lecid_pair][instance_id], lang)
                    case "slides":
                        src, gold, pred, gold_tokenized, pred_tokenized = \
                            _get_triplets_slide_(gold_all_langs['en'], gold_all_langs[lang], preds[sbj_lecid_pair][instance_id][lang], lang)
                        if not (len(src) == len(gold) == len(pred)):
                            src, gold, pred = map('\n'.join, [src, gold, pred])
                            gold_tokenized = [[__ for _ in gold_tokenized for __ in _]]
                            pred_tokenized = [[__ for _ in pred_tokenized for __ in _]]
                    case _:
                        raise NotImplementedError(f"Not implemented get_scores(task='{task}')")
                _n = min(5, max(map(len, pred_tokenized)))
                scores_per_span[lang] = [
                    sacrebleu.corpus_bleu(pred, [gold], **get_bleu_kwargs(lang)).score,
                    sacrebleu.corpus_chrf(pred, [gold]).score,
                    corpus_nist([[_] for _ in gold_tokenized], pred_tokenized, n=_n) if _n > 0 else .0,
                    # cer_metric.compute(predictions=[pred], references=[gold]),
                    # wer_metric.compute(predictions=[pred_tokenized], references=[gold_tokenized]),
                    # [scorer.score(g, p).fmeasure for g, p in zip(gold, pred)],
                ] if not compute_comet else []
                _scores_per_lang = scores_per_lang.setdefault(lang, dict())
                for k, v in [
                    ("pred", pred), ("gold", gold), ("src", src),
                    ("pred_tokenized", pred_tokenized), ("gold_tokenized", gold_tokenized),
                    ("instance_id", [instance_id]), ("instance_len", [len(gold)]),
                ]:
                    _scores_per_lang.setdefault(k, []).extend(v)        # micro-averaged within same instance_id
            scores_per_subject["per_instance"][instance_id] = scores_per_span

        scores_per_subject["per_language"] = {
            lang: ([
                sacrebleu.corpus_bleu(_d["pred"], [_d["gold"]], **get_bleu_kwargs(lang)).score,
                sacrebleu.corpus_chrf(_d["pred"], [_d["gold"]]).score,
                corpus_nist([[_] for _ in _d["gold_tokenized"]], _d["pred_tokenized"], n=5),
            ] if not compute_comet else [])
            for lang, _d in scores_per_lang.items()
        }

        if compute_comet:
            print("computing COMET")
            for lang, _d in scores_per_lang.items():
                score, score_qe = comet_predict(_d["pred"], _d["gold"], _d["src"], batch_size)
                scores_per_subject["per_language"][lang].append(score["system_score"])
                scores_per_subject["per_language"][lang].append(score_qe["system_score"])
                _cur_id, s, s_qe = 0, score["scores"], score_qe["scores"]
                for instance_id, instance_len in zip(_d["instance_id"], _d["instance_len"]):
                    _sps = scores_per_subject["per_instance"][instance_id][lang]
                    _sps.append(sum(s[_cur_id:_cur_id+instance_len])/instance_len)
                    _sps.append(sum(s_qe[_cur_id:_cur_id+instance_len])/instance_len)
                    _cur_id += instance_len

        scores_all[sbj_lecid_pair] = scores_per_subject
    return scores_all


In [ ]:
# %%time
# fd_outs_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/outs_transcripts"
# fd_evals_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/evals_transcripts"

compute_comet = bool(ast.literal_eval(os.getenv("__EVAL_USE_COMET__", '1')))
_bs_ = int(os.getenv("__EVAL_USE_COMET_BS__", 128))

# _suffix_name_ = os.getenv("__EVAL_SUFFIX_NAME__", "transcripts_annotd")
# _suffix_name_ = os.getenv("__EVAL_SUFFIX_NAME__", "transcripts_aligned")
# _suffix_name_ = os.getenv("__EVAL_SUFFIX_NAME__", "transcripts_asrraw")
_suffix_name_ = os.getenv("__EVAL_SUFFIX_NAME__", "slides_aligned")
_task_ = _suffix_name_.split("_", 1)[0]

_st_ = int(os.getenv("__EVAL_SPLIT_ST__", 0)) or None
_ed_ = int(os.getenv("__EVAL_SPLIT_ED__", 1)) or None

print(compute_comet, _suffix_name_, _task_, _st_, _ed_)

match _task_:
    case "transcripts":
        fp_reference = _get_basedir_path() / "25.11.transcript_translate/transcript_translate_annotated_by_sbj_all_text.json"
    case "slides":
        fp_reference = _get_basedir_path() / "25.11.slides_annot/slide_translation_annot_by_sbj_all_text.json"
    case _:
        raise NotImplementedError(f"Not implemented task='{_task_}'")
assert fp_reference.exists(), f"File not found: {fp_reference}"
with open(fp_reference, 'r', encoding="utf8") as f:
    golds = json.load(f)

fd_outs_basedir = _get_basedir_path() / f"lecture-translate/6.transcripts_translation/outs_{_suffix_name_}"
fd_evals_basedir = fd_outs_basedir.with_name(f"evals_{_suffix_name_}" + ("_comet" if compute_comet else ""))
fd_preds_list = sorted(filter(
    lambda fd: (
        # fd.name.startswith("o4-mini") and
        # fd.name.startswith("glm46v-flash") and
        fd.name.endswith(("nocontext", "slideimg", "slidetxt"))),
    fd_outs_basedir.iterdir()
    ))[_st_:_ed_]
print("Prediction List:", *[_.name for _ in fd_preds_list], sep="\n")
for _, fd_preds in enumerate(tqdm(fd_preds_list)):
    if not fd_preds.is_dir():
        continue
    preds = {}
    for k, v in golds.items():
        sbj, lec_id = k.split('/', 1)
        fp_preds = sorted(fd_preds.glob("*.json"))
        for fp in fp_preds:
            matching_file = (
                fp.name.startswith(f"{sbj}_{lec_id}") or
                (fd_preds.name.startswith(("gpt-4.1", "gpt-5-mini", "o4-mini"))
                 and fp.name.startswith(f"{sbj}_"))
            )
            if matching_file:
                assert k not in preds
                with open(fp, 'r', encoding="utf8") as f:
                    preds[k] = json.load(f)

    scores = get_scores(golds, preds, compute_comet=compute_comet, batch_size=_bs_, task=_task_)
    fd_evals_basedir.mkdir(exist_ok=True)
    with open(fd_evals_basedir / f"{fd_preds.name}.json", 'w', encoding="utf8") as f:
        json.dump(scores, f, ensure_ascii=False, indent=2)

In [ ]:
# ########################################################
# # Corpus score per subject != averaging over instances #
# ########################################################

# def _avg(l):
#     return sum(l) / len(l)

# for sbj_lecid_pair, _cur_scores in scores.items():
#     print(sbj_lecid_pair)
#     for lang, score in _cur_scores["per_language"].items():
#         concat = {}
#         for ts, _score in _cur_scores["per_instance"].items():
#             concat.setdefault("bleu", []).append(_score[lang][0])
#             concat.setdefault("chrf", []).append(_score[lang][1])
#         print(" ", lang, f"{score[0]:8.3f}, {_avg(concat['bleu']):8.3f}", f"{score[1]:8.3f}, {_avg(concat['chrf']):8.3f}")

---
##### Converting GPT results

In [ ]:
# %%time
# assert False, "Deprecated due to prompt change"
# fd_outs_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/outs_slidetextonly"
# fd_evals_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/evals_slides"
# for fd_preds in tqdm(sorted(fd_outs_basedir.iterdir())):
#     if not fd_preds.is_dir():
#         continue
#     preds = {}
#     for fp in sorted(fd_preds.glob("*.json")):
#         with open(fp, 'r', encoding="utf8") as f:
#             cur_pred = json.load(f)
#         assert len(cur_pred) == 1
#         assert list(cur_pred.keys())[0] not in preds

#         # Convert GPT dict_keys due to previous mishandling during query
#         _key = sorted(cur_pred.keys())[0]
#         _cur_pred = {}
#         for _key_pred in cur_pred[_key].keys():
#             for _key_gold in golds[_key].keys():
#                 if _key_gold.endswith(_key_pred):
#                     break
#             _cur_pred[_key_gold] = cur_pred[_key][_key_pred]
#         preds[_key] = _cur_pred

#     scores = get_scores(golds, preds)
#     fd_evals_basedir.mkdir(exist_ok=True)
#     with open(fd_evals_basedir / f"{fd_preds.name}_nocontext.json", 'w', encoding="utf8") as f:
#         json.dump(scores, f, ensure_ascii=False, indent=2)

Computing Scores

In [ ]:
# %%time
# fd_outs_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/outs_slides"
# fd_evals_basedir = _get_basedir_path() / "lecture-translate/6.transcripts_translation/evals_slides"
# for fd_preds in tqdm(sorted(fd_outs_basedir.iterdir())):
#     if not fd_preds.is_dir():
#         continue
#     preds = {}
#     for k, v in golds.items():
#         sbj, lec_id = k.split('/', 1)
#         fp_preds = sorted(fd_preds.glob("*.json"))
#         for fp in fp_preds:
#             if fp.name.startswith(f"{sbj}_{lec_id}"):
#                 assert k not in preds
#                 with open(fp, 'r', encoding="utf8") as f:
#                     preds[k] = json.load(f)
#         # True
#         # assert all(
#         #     sum(
#         #         fp.name.startswith('_'.join(sbj_lecid_pair.split('/', 1)))
#         #         for fp in fp_preds
#         #     ) == 1
#         #     for sbj_lecid_pair in golds.keys()
#         # )

#     scores = get_scores(golds, preds)
#     fd_evals_basedir.mkdir(exist_ok=True)
#     with open(fd_evals_basedir / f"{fd_preds.name}.json", 'w', encoding="utf8") as f:
#         json.dump(scores, f, ensure_ascii=False, indent=2)

---
---
---

##### Merging Evaluation Scores

In [ ]:
fd_evals_basedir = Path("evals_transcripts_annotd")
fd_evals_basedir_comet = Path("evals_transcripts_annotd_comet")
fd_score_transcripts = Path("scores_transcripts_annotd")

In [ ]:
fd_evals_basedir = Path("evals_transcripts_aligned")
fd_evals_basedir_comet = Path("evals_transcripts_aligned_comet")
fd_score_transcripts = Path("scores_transcripts_aligned")

In [ ]:
fd_evals_basedir = Path("evals_transcripts_asrraw")
fd_evals_basedir_comet = Path("evals_transcripts_asrraw_comet")
fd_score_transcripts = Path("scores_transcripts_asrraw")

In [ ]:
fd_evals_basedir = Path("evals_slides_aligned")
fd_evals_basedir_comet = Path("evals_slides_aligned_comet")
fd_score_transcripts = Path("scores_slides_aligned")

In [ ]:
# Merging Evaluation Scores
for fp in sorted(fd_evals_basedir.glob("*.json")):
    print(fp.name, (fd_evals_basedir_comet / fp.name).exists())
    with open(fp, 'r', encoding="utf8") as f:
        scores = json.load(f)
    with open(fd_evals_basedir_comet / fp.name, 'r', encoding="utf8") as f:
        scores_comet = json.load(f)
    concat_scores = {
        sbj: {
            "per_language": {
                lang: res_lang + scores_comet[sbj]["per_language"][lang]
                for lang, res_lang in res["per_language"].items()
            },
            "per_instance": {
                ts: {
                    lang: res_lang + scores_comet[sbj]["per_instance"][ts][lang]
                    for lang, res_lang in res.items()
                }
                for ts, res in res["per_instance"].items()
            },
        }
        for sbj, res in scores.items()
    }
    fd_score_transcripts.mkdir(parents=True, exist_ok=True)
    with open(fd_score_transcripts / fp.name, 'w', encoding="utf8") as f:
        json.dump(concat_scores, f, ensure_ascii=False, indent=2)

---
---
---

##### For Writings

In [ ]:
"""
cp -rd preds_aya_vision_8b__260517_201324_451981    ../outs_transcripts_aligned/aya-v-8b_nocontext
cp -rd preds_aya_vision_8b_slideimg_260517_201329_451974    ../outs_transcripts_aligned/aya-v-8b_slideimg
cp -rd preds_aya_vision_8b_slidetxt_260517_201329_451982    ../outs_transcripts_aligned/aya-v-8b_slidetxt
cp -rd preds_qwen35_9b__260517_145928_450617    ../outs_transcripts_aligned/qwen35-9b_nocontext
cp -rd preds_qwen35_9b_slideimg_260517_145924_450607    ../outs_transcripts_aligned/qwen35-9b_slideimg
cp -rd preds_qwen35_9b_slidetxt_260517_145928_450618    ../outs_transcripts_aligned/qwen35-9b_slidetxt

cp -rd preds_aya_vision_8b__260517_201329_451978    ../outs_transcripts_annotd/aya-v-8b_nocontext
cp -rd preds_aya_vision_8b_slideimg_260517_201329_451980    ../outs_transcripts_annotd/aya-v-8b_slideimg
cp -rd preds_aya_vision_8b_slidetxt_260517_201329_451979    ../outs_transcripts_annotd/aya-v-8b_slidetxt
cp -rd preds_qwen35_9b__260517_201458_451975    ../outs_transcripts_annotd/qwen35-9b_nocontext
cp -rd preds_qwen35_9b_slideimg_260517_201457_451977    ../outs_transcripts_annotd/qwen35-9b_slideimg
cp -rd preds_qwen35_9b_slidetxt_260517_201458_451976    ../outs_transcripts_annotd/qwen35-9b_slidetxt

cp -rd preds_llama4scout__260308_175744    ../outs_transcripts_annotd/llama4scout_nocontext
cp -rd preds_llama4scout_slideimg_260312_012602    ../outs_transcripts_annotd/llama4scout_slideimg
cp -rd preds_llama4scout_slideimg_260312_230811/*    ../outs_transcripts_annotd/llama4scout_slideimg
cp -rd preds_llama4scout_slidetxt_260309_225607    ../outs_transcripts_annotd/llama4scout_slidetxt
"""

In [ ]:
lang_code_to_name = {
    # 'en': 'English',
    'ja': 'Japanese',
    'zh': 'Chinese',
    'th': 'Thai',
    'id': 'Indonesian',
    'es': 'Spanish',
    'fr': 'France',
    'de': 'German',
}
models_to_shortname = {
    'llama4scout': 'llama4scout',
    'llama32': 'llama32',
    'gemma-3-12b-it': 'gemma-3',
    'gpt-4.1-2025-04-14': 'gpt-41',
    'gpt-5-mini-2025-08-07': 'gpt-5-mini',
    'o4-mini-2025-04-16': 'o4-mini',
    'qwen35-9b': 'qwen35-9b',
    'aya-v-8b': 'aya-v-8b',
    'internvl35-8b': 'internvl35-8b',
    'glm46v-flash': 'glm46v-flash',
}
model_shortname_to_longname = {
    'llama4scout': 'Llama 4 Scout',
    'llama32': "Llama 3.2 11B",
    'gemma-3': "Gemma-3 12B",
    'gpt-41': "GPT-4.1",
    'gpt-5-mini': "GPT-5 mini",
    'o4-mini': "o4-mini",
    # 'qwen35-9b': 'Qwen 3.5 9B',
    # 'aya-v-8b': 'Aya Vision 8B',
    # 'internvl35-8b': 'InternVL 3.5 8B',
    # 'glm46v-flash': 'GLM 4.6V Flash',
}
rename_subject_map = {
    "intmd_microeconomics": "i-microecons",
    "intro_to_cs_programming_python": "i-cs-prog",
    "intro_to_neural_computation": "i-neural-comp",
    "intro_to_physics_20century": "i-physics",
    "theory_of_computation": "theory-comp",
    "urban_energy_systems_and_policy": "urban-policy",
}

In [ ]:
def format_col(col, precision=3):
    top2 = col.nlargest(2)
    def fmt(x):
        x_str = f"{x:.{precision}f}"
        if x == top2.iloc[0]:
            return f"\\textbf{{{x_str}}}"
        elif x == top2.iloc[1]:
            return f"\\underline{{{x_str}}}"
        else:
            return x_str
    return col.map(fmt)

def get_latex(cur, row_wise=True, **kwargs):
    if row_wise:
        df_fmt = cur.copy()
        for c in cur.columns:
            df_fmt[c] = format_col(cur[c], precision=1)
    else:
        df_fmt = cur.T.copy()
        for c in cur.T.columns:
            df_fmt[c] = format_col(cur.T[c], precision=1)
        df_fmt = df_fmt.T
    return df_fmt.to_latex(escape=False, **kwargs)

---
---
---

Consolidating Tables

In [ ]:
# import os
# os.chdir("6.transcripts_translation")

In [ ]:
from pathlib import Path
from constants import LANGUAGES
import json
import pandas as pd
import seaborn as sns

In [ ]:
# fd_score_transcripts = Path("scores_transcripts")
fd_score_transcripts = Path("scores_transcripts_asrraw")
# fd_score_transcripts = Path("scores_transcripts_aligned")
# fd_score_transcripts = Path("scores_transcripts_annotd")

In [ ]:
def get_scores(suffix):
    scores = {}
    for fp in sorted(fd_score_transcripts.glob(f"*{suffix}")):
        model_name = models_to_shortname[fp.name[:-len(suffix)]]
        with open(fp, 'r', encoding="utf8") as f:
            scores[model_name] = json.load(f)
        if hasattr(scores[model_name], "model"):
            assert scores[model_name].model == model_name
    return scores

In [ ]:
# # Macro average score
# def avg(l):
#     return sum(l) / len(l)

# scores_overall = {}
# all_scores = []
# for model_name, model_scores in scores.items():
#     scores_all_sbj_cur_model = {}
#     for target_lang in filter(lambda l: l != 'en', LANGUAGES.keys()):
#         bleu, chrf = [], []
#         for sbj_lecid_pair in model_scores.keys():
#             _scores = [s[target_lang] for s in model_scores[sbj_lecid_pair].values()]
#             _bleu, _chrf = map(avg, zip(*_scores))
#             all_scores.append((model_name, target_lang, sbj_lecid_pair, _chrf, _bleu))
#             bleu.append(_bleu)
#             chrf.append(_chrf)
#         scores_all_subjects_cur_model[target_lang] = (avg(bleu), avg(chrf))
#     scores_overall[model_name] = scores_all_sbj_cur_model


In [ ]:
def get_scores_df(suffix):
    # global model_name, model_scores
    scores = get_scores(suffix)
    cur_items = []
    for model_name, model_scores in scores.items():
        for sbj_lecid_pair, cur_scores in model_scores.items():
            sbj, lec_id = sbj_lecid_pair.split('/', 1)
            for target_lang in filter(lambda l: l != 'en', LANGUAGES.keys()):
                bleu_s, chrf_s, nist_s, comet_s, cometkiwi_s = cur_scores["per_language"][target_lang]
                cur_items.append((
                    model_shortname_to_longname[model_name],
                    rename_subject_map.get(sbj, sbj),
                    target_lang,
                    chrf_s,
                    bleu_s,
                    nist_s,
                    comet_s*100,
                    cometkiwi_s*100,
                ))
    return pd.DataFrame(cur_items, columns=["model", "sbj", "lang", "ChrF", "BLEU", "NIST", "COMET", "COMETKIWI"])

In [ ]:
def get_scores_df_per_subject(suffix, metrics="BLEU_score"):
    scores, _ = get_scores(suffix)
    cur_items = []
    for model_name, model_scores in scores.items():
        for target_lang in filter(lambda l: l != 'en', LANGUAGES.keys()):
            for sbj_lecid_pair, d in model_scores["results"][target_lang]["per_subjects"].items():
                sbj, lec_id = sbj_lecid_pair.split('/', 1)
                cur_items.append((
                    model_shortname_to_longname[model_name],
                    target_lang,
                    rename_subject_map.get(sbj, sbj).replace('_', '-'),
                    lec_id,
                    d[metrics],
                ))
    return pd.DataFrame(cur_items, columns=["model", "lang", "sbj", "lec_id", "score"])

In [ ]:
# for suffix in ["_nocontext.json", "_slidetext.json", "_slideimage.json"]:
for suffix in ["_nocontext.json", "_slidetxt.json", "_slideimg.json"]:
# for suffix in ["_nocontext.json"]:
# for suffix in ["_slidetxt.json", "_slideimg.json"]:
    df = get_scores_df(suffix)
    gb = df.groupby("model")
    _cur = pd.DataFrame({
        m: (d.ChrF.mean(), d.BLEU.mean(), d.COMET.mean(), d.COMETKIWI.mean(), d.NIST.mean()) for m, d in gb
    }).T.reset_index().rename(columns={"index": "model", 0: "ChrF", 1: "BLEU", 2: "COMET", 3: "KIWI", 4: "NIST"})
    cur = _cur.set_index("model").loc[model_shortname_to_longname.values()]
    print('%', suffix)
    print('\n '.join(get_latex(cur).split('\n')).replace("\\textbf{0.0}", "0.0"), end="\n\n")

In [ ]:
# Result per Language
for suffix in ["_nocontext.json", "_slidetxt.json", "_slideimg.json"]:
    df = get_scores_df(suffix)
    cur = df.pivot_table(index="model", columns="lang", values="NIST").loc[model_shortname_to_longname.values(),lang_code_to_name.keys()]
    cur.index = cur.index.map(lambda t: f"{{\\small {t}}}")
    print('%', suffix)
    print('\n '.join(get_latex(cur).split('\n')).replace("\\textbf{0.0}", "0.0"), end="\n\n")

In [ ]:
# Result per Subject
for suffix in ["_nocontext.json", "_slidetxt.json", "_slideimg.json"]:
    df = get_scores_df(suffix)
    cur = df.pivot_table(index="model", columns="sbj", values="NIST").loc[model_shortname_to_longname.values()]
    cur.index = cur.index.map(lambda t: f"{{\\small {t}}}")
    print('%', suffix)
    print('\n'.join(get_latex(cur).split('\n')).replace("\\textbf{0.0}", "0.0"), end="\n\n")

In [ ]:
for suffix in ["_nocontext.json", "_slidetext.json", "_slideimage.json"]:
    df = get_scores_df_per_subject(suffix)
    _cur = df.groupby(["model", "sbj"]).score.mean().reset_index().pivot(
        index="sbj", columns="model", values="score").loc[:,model_shortname_to_longname.values()]
    # print('\n & '.join(get_latex(cur).split('\n')), end="\n\n")
    print(suffix, '\n')
    print(get_latex(_cur, row_wise=False, index=(suffix=="_nocontext.json")), end= "\n\n\n")


In [ ]:
_cur.index

---

In [ ]:
import os
os.chdir("6.transcripts_translation")

In [ ]:
from pathlib import Path
from constants import LANGUAGES
import json
import pandas as pd
import seaborn as sns

In [ ]:
fd_score_slides = Path("scores_slides")

In [ ]:
list(fd_score_slides.glob(f"*_slideimg.json"))

In [ ]:
list(fd_score_slides.glob(f"*_nocontext.json"))

In [ ]:
def get_scores(suffix):
    scores = {}
    for fp in sorted(fd_score_slides.glob(f"*{suffix}")):
        model_name = models_to_shortname[fp.name[:-len(suffix)]]
        with open(fp, 'r', encoding="utf8") as f:
            scores[model_name] = json.load(f)
    return scores

In [ ]:
def get_scores_df(suffix):
    scores = get_scores(suffix)
    cur_items = []
    for model_name, model_scores in scores.items():
        for sbj_lecid_pair in model_scores.keys():
            sbj, lec_id = sbj_lecid_pair.split('/', 1)
            for scene_id, scene_scores in model_scores[sbj_lecid_pair].items():
                for target_lang in filter(lambda l: l != 'en', LANGUAGES.keys()):
                    _bleu, _chrf = scene_scores[target_lang]
                    cur_items.append((
                        model_shortname_to_longname[model_name],
                        rename_subject_map.get(sbj, sbj).replace('_', '-'),
                        lec_id,
                        scene_id,
                        target_lang,
                        _chrf,
                        _bleu,
                    ))

    return pd.DataFrame(cur_items, columns=["model", "sbj", "lec_id", "scene_id", "lang", "ChrF", "BLEU"])

In [ ]:
for suffix in ["_nocontext.json", "_slideimg.json"]:
    df = get_scores_df(suffix)
    df_per_subject = pd.concat([
        df.groupby(["model", "sbj", "lec_id", "lang"]).ChrF.mean(),
        df.groupby(["model", "sbj", "lec_id", "lang"]).BLEU.mean(),
    ], axis=1).reset_index()
    cur = pd.concat([
        df_per_subject.groupby(["model"]).ChrF.mean(),
        df_per_subject.groupby(["model"]).BLEU.mean(),
    ], axis=1)
    # if suffix == "_nocontext.json":    # as other suffix is not completed yet
    #     cur = cur.loc[model_shortname_to_longname.values()]
    cur = cur.loc[model_shortname_to_longname.values()]
    print(suffix, '\n')
    print(get_latex(cur), end="\n\n\n")

In [ ]:
for suffix in ["_nocontext.json"]:
    df = get_scores_df(suffix)
    df_per_subject = pd.concat([
        df.groupby(["model", "sbj", "lec_id", "lang"]).ChrF.mean(),
        df.groupby(["model", "sbj", "lec_id", "lang"]).BLEU.mean(),
    ], axis=1).reset_index()
    cur = pd.concat([
        df_per_subject.groupby(["model", "sbj"]).ChrF.mean(),
        df_per_subject.groupby(["model", "sbj"]).BLEU.mean(),
    ], axis=1).reset_index().pivot(
        index="sbj", columns="model", values=["ChrF"]
    )
    print(suffix, '\n')
    print(get_latex(cur, row_wise=False, index=True), end="\n\n\n")

In [ ]:
for suffix in ["_nocontext.json"]:
    df = get_scores_df(suffix)
    df_per_subject = pd.concat([
        df.groupby(["model", "sbj", "lec_id", "lang"]).ChrF.mean(),
        df.groupby(["model", "sbj", "lec_id", "lang"]).BLEU.mean(),
    ], axis=1).reset_index()
    cur = pd.concat([
        df_per_subject.groupby(["model", "sbj"]).ChrF.mean(),
        df_per_subject.groupby(["model", "sbj"]).BLEU.mean(),
    ], axis=1).reset_index().pivot(
        index="sbj", columns="model", values=["BLEU"]
    )
    print(suffix, '\n')
    print(get_latex(cur, row_wise=False, index=False), end="\n\n\n")

In [ ]:
df = get_scores_df_per_subject(suffix)
_cur = df.groupby(["model", "sbj"]).score.mean().reset_index().pivot(
    index="sbj", columns="model", values="score").loc[:,model_shortname_to_longname.values()]
# print('\n & '.join(get_latex(cur).split('\n')), end="\n\n")
print(suffix, '\n')
print(get_latex(_cur, row_wise=False, index=(suffix=="_nocontext.json")), end= "\n\n\n")


In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns